<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Install required libraries

This installs the core Python libraries used in the notebook for data manipulation and plotting.

In [ ]:
!pip install TA-Lib pandas matplotlib

TA-Lib requires a separate C library installed on your system before the Python wrapper will work. Install it via conda (conda install -c conda-forge ta-lib) or follow the platform-specific build instructions at github.com/TA-Lib/ta-lib-python. The openbb-terminal SDK also has its own installation process; see openbb.co/docs for setup steps.

## Imports and setup

matplotlib renders our charts inline in the notebook. pandas handles tabular price data. talib provides optimized C-based implementations of Bollinger Bands, RSI, and MACD. openbb_terminal supplies a unified API for pulling stock data and a theming utility for chart styling.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from talib import BBANDS, MACD, RSI
import yfinance as yf

We import all libraries up front so the rest of the notebook can focus on analysis rather than setup. TA-Lib is the same library used on many professional desks, so the indicator values we compute here match what other traders see on their screens. That alignment is the whole point, as the post explains: indicators matter because the crowd watches them.

## Load price data for analysis

We pull AAPL daily data starting from 2020 and rename the adjusted close column to "close" so it works cleanly with TA-Lib's functions.

In [ ]:
data = yf.download("AAPL", start="2020-01-01", multi_level_index=False)

Using adjusted close rather than raw close accounts for stock splits and dividends, giving us a price series that reflects actual returns. Starting from 2020 gives us several years of data, enough to see how these indicators behave across both trending and choppy markets. This is the same kind of historical dataset you would use to backtest a strategy before risking real money.

## Compute three core technical indicators

Bollinger Bands wrap a 21-day moving average with bands set two standard deviations above and below. When price touches or exceeds the upper band, the stock may be overbought relative to its recent range.

In [ ]:
up, mid, low = BBANDS(
    data.Close,
    timeperiod=21,
    nbdevup=2,
    nbdevdn=2,
    matype=0,
)

The 21-day window and 2-standard-deviation width are the most common defaults across trading platforms. Because so many traders use these exact settings, the bands become self-reinforcing levels where buying and selling pressure tends to cluster.

RSI condenses the last 14 days of gains and losses into a single number between 0 and 100. Readings above 70 suggest overbought conditions, while readings below 30 suggest oversold.

In [ ]:
rsi = RSI(data.Close, timeperiod=14)

RSI gives us a momentum check that is independent of the Bollinger Bands. Professionals rarely act on one indicator alone. When RSI confirms what the bands are showing, the signal carries more weight because two different formulas are pointing in the same direction.

MACD compares a 12-day and 26-day exponential moving average, then smooths the difference with a 9-day signal line. Crossovers between the MACD line and its signal line highlight shifts in trend direction.

In [ ]:
macd, macdsignal, macdhist = MACD(
    data.Close,
    fastperiod=12,
    slowperiod=26,
    signalperiod=9,
)

The histogram (macdhist) shows the gap between the MACD line and its signal line. When that gap widens, momentum is accelerating. When it shrinks toward zero, the trend may be about to reverse. This is the kind of early warning that Eddie tracked in his spreadsheet.

## Organize and plot the indicators

We assemble the MACD components into their own DataFrame so they can be plotted together on a single axis with a shared legend.

In [ ]:
macd = pd.DataFrame(
    {
        "MACD": macd,
        "MACD Signal": macdsignal,
        "MACD History": macdhist,
    }
)

We combine the price, Bollinger Bands, and RSI into one DataFrame. Keeping everything aligned by date index ensures the plots stay synchronized across all three subplots.

In [ ]:
data = pd.DataFrame(
    {
        "AAPL": data.Close,
        "BB Up": up,
        "BB Mid": mid,
        "BB down": low,
        "RSI": rsi,
    }
)

This three-panel chart stacks Bollinger Bands, RSI, and MACD vertically with a shared time axis so you can visually cross-reference all three indicators at any point in time.

In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    figsize=(15, 10),
    sharex=True,
)
data.drop(["RSI"], axis=1).plot(
    ax=axes[0],
    lw=1,
    title="BBANDS",
)
data["RSI"].plot(
    ax=axes[1],
    lw=1,
    title="RSI",
)
axes[1].axhline(70, lw=1, ls="--", c="k")
axes[1].axhline(30, lw=1, ls="--", c="k")
macd.plot(
    ax=axes[2],
    lw=1,
    title="MACD",
    rot=0,
)
axes[2].set_xlabel("")
fig.tight_layout()
plt.show()

The dashed lines at 70 and 30 on the RSI panel mark the conventional overbought and oversold thresholds. Look for moments where price touches the upper Bollinger Band while RSI is above 70 and the MACD histogram is shrinking. Those are the spots where multiple indicators agree that a pullback may be coming, exactly the kind of confluence that traders like Eddie watched for before placing a trade.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.